In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
staging_prospect=f"{catalog_name}.staging.prospect_current"
silver_batchdate=f"{catalog_name}.silver.batchdate"
gold_prospect=f"{catalog_name}.gold.dim_prospect"
gold_customer=f"{catalog_name}.gold.dim_customer"

In [0]:
batch_id = dbutils.widgets.get("batch_id")

In [0]:
df_staging=spark.read.table(staging_prospect)
# df_customer=spark.read.table(gold_customer)
df_batchdate=spark.read.table(silver_batchdate)

In [0]:
# df_staging.printSchema()

In [0]:
# concatination_logic="concat(upper(lastname),upper(firstname),upper(addressline1),coalesce(upper(addressline2),''),upper(postalcode))"
concatination_logic="concat(upper(LastName),upper(FirstName),upper(AddressLine1),coalesce(upper(AddressLine2),''),upper(PostalCode))"

In [0]:
df_staging=df_staging.withColumn("match_key",expr(concatination_logic))
# df_customer=df_customer.withColumn("match_key",expr(concatination_logic))


In [0]:
# spark.catalog.tableExists(gold_customer)
# spark.catalog.tableExists(gold_prospect)

In [0]:
if spark.catalog.tableExists(gold_customer):
    df_customer=spark.table(gold_customer).filter(col("iscurrent")==True)

    cust_concat_logic="concat(upper(lastname),upper(firstname),upper(addressline1),coalesce(upper(addressline2),''),upper(postalcode))"

    df_customer=df_customer.withColumn("match_key",expr(cust_concat_logic))

    df_match_cust=df_customer.select("match_key").dropDuplicates().withColumn("is_matched",lit(True))

    df_staging=df_staging.join(df_match_cust,"match_key","left")\
    .withColumn("iscustomer",when(col("is_matched")==True,lit(True)).otherwise(lit(False)))
else:
    df_staging=df_staging.withColumn("iscustomer",lit(False))

In [0]:
# #get unqiue matched customer keys
# df_match_cust=df_customer.select("match_key").dropDuplicates().withColumn("is_matched",lit(True))

In [0]:
# Left join to get flag customer
# df_staging=df_staging.join(df_match_cust,"match_key","left")\
#     .withColumn("iscustomer",when(col("is_matched")==True,True).otherwise(False))

In [0]:
# df_staging.printSchema()

In [0]:
df_staging=df_staging.withColumn("marketingnameplate",
                    when((col("NetWorth") > 1000000) | (col("Income")>200000),lit("High Net Worth"))\
                    .when((col("Age")>60)|(col("CreditRating") > 750), lit("Retirement"))\
                    .when(col("age")<35,lit("Young Professional"))\
                    .otherwise(lit("Standard")))

In [0]:
df_batch_sk=df_batchdate.withColumn("date_sk",expr("cast(date_format(batchdate,'yyyyMMdd') as bigint)"))


In [0]:
# df_batch_sk = df_batch_sk.withColumn(
#     "_batch",regexp_extract(col("_batch_id"), r'(\d+)', 1)
# ).drop("_batch_id")
# df_batch_sk = df_batch_sk.withColumn(
#     "_batch",col("_batch")
# )

In [0]:
# df_batch_sk.printSchema()

In [0]:
current_date_sk_row=df_batch_sk.filter(col("_batch")==batch_id).select("date_sk").collect()
current_date_sk=current_date_sk_row if current_date_sk_row else None

In [0]:
df_staging=df_staging.join(df_batch_sk.select(col("_batch").alias("first_batchid"),col("date_sk").alias("sk_updatedateid")),"first_batchid","left")

In [0]:
# df_staging.printSchema()

In [0]:
df_gold = df_staging.select(col("AgencyID").alias("agencyid"),
    lit(current_date_sk[0][0]).cast("BIGINT").alias("sk_recorddateid"),
    col("sk_updatedateid"),col("first_batchid").alias("batch"),
    col("iscustomer"),col("LastName").alias("lastname"),
    col("FirstName").alias("firstname"),
    col("MiddleInitial").alias("middleinitial"),
    col("Gender").alias("gender"),
    col("AddressLine1").alias("addressline1"),
    col("AddressLine2").alias("addressline2"),
    col("PostalCode").alias("postalcode"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Country").alias("country"),
    col("Phone").alias("phone"),
    col("Income").alias("income"),
    col("NumberCars").alias("numbercars"),
    col("NumberChildren").alias("numberchildren"),
    col("MaritalStatus").alias("maritalstatus"),
    col("Age").alias("age"),
    col("CreditRating").alias("creditrating"),
    col("OwnOrRentFlag").alias("ownorrentflag"),
    col("Employer").alias("employer"),
    col("NumberCreditCards").alias("numbercreditcards"),
    col("NetWorth").alias("networth"),
    col("marketingnameplate")
)

In [0]:
# df_gold.limit(10).display()
# df_gold.filter(col("iscustomer")==True).display()
# df_gold.count()

In [0]:
try:
    #Gold Table INSERT OVERWRITE for Current state only
    print("OverWriting to gold table")
    df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_prospect)
    print("Write successfully")


    run_id=df_gold.select("_run_id").first()[0]
    staging_history = spark.sql(f"DESCRIBE HISTORY {staging_prospect}").first()
    source_count = int(staging_history["operationMetrics"].get("numOutputRows", 0))

    gold_history = spark.sql(f"DESCRIBE HISTORY {gold_prospect}").first()
    target_count = int(gold_history["operationMetrics"].get("numOutputRows", 0))


    log_pipeline_recon(
            spark=spark,
            run_id=run_id,
            batch_id=batch_id,
            domain="CUSTOMER",
            table_name="dim_prospect",  
            source_layer="staging", 
            target_layer="gold",
            source_count=source_count,
            target_count=target_count
    )
    
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch=batch_id, 
        layer="gold",
        table_name="dim_prospect",
        operation="OVERWRITE",      # Architecture explicitly requires OVERWRITE for this dimension
        rows_affected=target_count
    )
except Exception as e:
    print(e)

In [0]:
spark.table(gold_prospect).count()